# VisionTrack AI – Phase 1 training (Google Colab, Tesla T4)
Optic disc & cup segmentation (U-Net) · CDR estimation · Glaucoma classification (EfficientNet-B0)

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Get the project code
Upload `VisionTrackAI.zip` to the root of your Google Drive (MyDrive), then run:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!unzip -q -o /content/drive/MyDrive/VisionTrackAI.zip -d /content/
%cd /content/VisionTrackAI
!pip -q install -r requirements.txt
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

## 2. Download the dataset (SMDG-19 – Multichannel Glaucoma Benchmark Dataset)
1. On kaggle.com → Settings → **Create New API Token** → downloads `kaggle.json`.
2. Run the cell and upload `kaggle.json` when asked.

In [ ]:
from google.colab import files
import os
if not os.path.exists('/root/.kaggle/kaggle.json'):
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.replace('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
!kaggle datasets download -d deathtrooper/multichannel-glaucoma-benchmark-dataset -p /content/smdg_zip
!mkdir -p data/smdg && unzip -q -o /content/smdg_zip/*.zip -d data/smdg
!ls data/smdg

## 3. Build train / val / test splits (70 / 15 / 15)

In [ ]:
!python scripts/prepare_data.py --data data/smdg

## 4. Train Component 1 – U-Net optic disc & optic cup segmentation (~1–2 h on T4)

In [ ]:
!python scripts/train_segmentation.py --epochs 40 --batch 8 --workers 2

## 5. Train glaucoma classifier – EfficientNet-B0 (~1 h on T4)

In [ ]:
!python scripts/train_classifier.py --epochs 25 --batch 24 --workers 2

## 6. Evaluate on the test set (tunes thresholds on validation first)
Produces Dice, CDR MAE / R², accuracy, sensitivity, specificity, ROC-AUC, plots and the failure-analysis report.

In [ ]:
!python scripts/evaluate.py --tune
!cat reports/phase1_results.json
!cat reports/failure_analysis.md

In [ ]:
from IPython.display import Image, display
for f in ['seg_training_curves.png','cls_training_curves.png','cdr_scatter.png','roc_curve.png','confusion_matrix.png','worst_cases.png']:
    p = f'reports/{f}'
    if os.path.exists(p): print(f); display(Image(p, width=600))

## 7. Demo figure – original / ground truth / prediction (like the PPT demonstration slide)

In [ ]:
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, '.')
from src.inference import VisionTrackPipeline, make_overlay
from src.preprocessing import read_rgb, read_mask
pipe = VisionTrackPipeline()
df = pd.read_csv('data/splits/seg_test.csv').sample(3, random_state=1)
for _, r in df.iterrows():
    rgb = read_rgb(r.fundus); gd = np.maximum(read_mask(r.disc, rgb.shape), read_mask(r.cup, rgb.shape)); gc = read_mask(r.cup, rgb.shape)
    res, d, c = pipe.analyze(rgb, r['name'])
    fig, ax = plt.subplots(2, 3, figsize=(12, 8))
    for a, im, t in zip(ax.ravel(), [rgb, gd, gc, d, c, make_overlay(rgb, d, c)],
                        ['Original Fundus', 'Ground Truth Disc', 'Ground Truth Cup', 'Predicted Disc', 'Predicted Cup', 'Predicted Disc + Cup']):
        a.imshow(im); a.set_title(t); a.axis('off')
    k = res['classification'] or {}
    fig.suptitle(f"{r['name']}  |  vCDR {res['component1']['cdr']['vertical_cdr']}  |  glaucoma prob {k.get('glaucoma_probability')}")
    plt.show()

## 8. (Optional) Visual-field matched dataset for Phase 2 – PAPILA
Download PAPILA (Kovalyk et al., Scientific Data 2022 – linked from the paper, hosted on figshare), unzip into `data/papila`, then:

In [ ]:
# !python scripts/prepare_papila_vf.py --papila data/papila

## 9. Save trained models and reports to Drive
Download these to your laptop and put them in the project's `checkpoints/` and `reports/` folders to run the Flask app.

In [ ]:
!mkdir -p /content/drive/MyDrive/VisionTrackAI_outputs
!cp -r checkpoints reports /content/drive/MyDrive/VisionTrackAI_outputs/
!ls -la /content/drive/MyDrive/VisionTrackAI_outputs/checkpoints